In [ ]:
import sys, itertools, json
import numpy as np, pandas as pd
from scipy.stats import chi2
 
DATA = sys.argv[1]  # directory holding the csv files
rng = np.random.default_rng(42)
COLS = ["aspect_ratio", "extent", "roundness", "circularity", "convexity", "solidity"]

In [ ]:
def zscore(X):
    return (X - X.mean(0)) / X.std(0, ddof=0)  # StandardScaler uses ddof=0
 
 
# ---------- load + outlier screening exactly as in the author's script ----------
df = pd.read_csv("basic_shape_features.csv")
Z = zscore(df[COLS].values)
cov = np.cov(Z, rowvar=False)
inv = np.linalg.inv(cov)
d = np.sqrt(np.einsum("ij,jk,ik->i", Z - Z.mean(0), inv, Z - Z.mean(0)))
thr = np.sqrt(chi2.ppf(1 - 0.001, df=6))
keep = d < thr
clean = df[keep].copy().reset_index(drop=True)
print("n all:", len(df), "| removed:", int((~keep).sum()), "| final:", len(clean))
print(clean.country.value_counts().to_dict())
 
clean["image_key"] = clean.filename.str.split("_leaf").str[0]
clean["origin"] = np.where(clean.filename.str.contains("SANAD"), "field", "repository")
print("\nLeaves by country x origin (n=442, filename convention):")
print(pd.crosstab(clean.country, clean.origin))
print("\nDistinct image keys per country (n=442):")
print(clean.groupby("country").image_key.nunique().to_dict())
print("leaves per image key:", clean.groupby("image_key").size().value_counts().sort_index().to_dict())

In [ ]:
# ---------- PERMANOVA (Euclidean, Anderson 2001) and PERMDISP ----------
def sq_dist(X):
    s = (X**2).sum(1)
    D2 = s[:, None] + s[None, :] - 2 * X @ X.T
    np.fill_diagonal(D2, 0)
    return np.maximum(D2, 0)
 
 
def pseudo_F_batch(D2, labels_perm, k):
    """labels_perm: (P, N) int array of group codes; returns pseudo-F for each row."""
    N = D2.shape[0]
    SST = D2.sum() / (2 * N)
    P = labels_perm.shape[0]
    SSW = np.zeros(P)
    for g in range(k):
        mask = (labels_perm == g)  # (P, N)
        ng = mask.sum(1)
        # sum of D2 within group g = m^T D2 m
        SSW += np.einsum("pi,ij,pj->p", mask.astype(float), D2, mask.astype(float)) / (2 * ng)
    SSA = SST - SSW
    return (SSA / (k - 1)) / (SSW / (N - k))
 
 
def permanova(X, groups, perms=999, seed=42):
    g, codes = np.unique(groups, return_inverse=True)
    k, N = len(g), len(codes)
    D2 = sq_dist(X)
    r = np.random.default_rng(seed)
    obs = pseudo_F_batch(D2, codes[None, :], k)[0]
    perm_codes = np.array([r.permutation(codes) for _ in range(perms)])
    Fp = pseudo_F_batch(D2, perm_codes, k)
    p = (np.sum(Fp >= obs) + 1) / (perms + 1)
    R2 = obs * (k - 1) / (obs * (k - 1) + (N - k))
    return obs, R2, p
 
 
def spatial_median(Y, iters=200, tol=1e-9):
    m = Y.mean(0)
    for _ in range(iters):
        dd = np.linalg.norm(Y - m, axis=1)
        dd = np.where(dd < 1e-12, 1e-12, dd)
        w = 1 / dd
        m_new = (Y * w[:, None]).sum(0) / w.sum()
        if np.linalg.norm(m_new - m) < tol:
            break
        m = m_new
    return m
 
 
def anova_F(dist, codes, k):
    N = len(dist)
    grand = dist.mean()
    ssb = sum((codes == g).sum() * (dist[codes == g].mean() - grand) ** 2 for g in range(k))
    ssw = sum(((dist[codes == g] - dist[codes == g].mean()) ** 2).sum() for g in range(k))
    return (ssb / (k - 1)) / (ssw / (N - k))
 
 
def permdisp(X, groups, perms=999, seed=42, center="median"):
    g, codes = np.unique(groups, return_inverse=True)
    k = len(g)
    dist = np.empty(len(X))
    for j in range(k):
        Y = X[codes == j]
        c = spatial_median(Y) if center == "median" else Y.mean(0)
        dist[codes == j] = np.linalg.norm(Y - c, axis=1)
    obs = anova_F(dist, codes, k)
    r = np.random.default_rng(seed)
    cnt = 0
    for _ in range(perms):
        if anova_F(dist, r.permutation(codes), k) >= obs:
            cnt += 1
    return obs, (cnt + 1) / (perms + 1)

In [ ]:
# ---------- 1. reproduce the manuscript numbers ----------
print("\n=== 1. REPRODUCTION (n=442, 5 countries) ===")
X = zscore(clean[COLS].values)
F, R2, p = permanova(X, clean.country.values, perms=999)
print(f"PERMANOVA  pseudo-F={F:.2f}  R2={R2:.4f}  p={p:.3f}   (manuscript: 9.81, 0.0824, 0.001)")
for c in ("median", "centroid"):
    Fd, pd_ = permdisp(X, clean.country.values, center=c)
    print(f"PERMDISP[{c}] F={Fd:.3f} p={pd_:.3f}   (manuscript: F=0.529, p=0.718)")
 
# pairwise (standardised once on full 442, as in the author's script)
rows = []
for a, b in itertools.combinations(sorted(clean.country.unique()), 2):
    m = clean.country.isin([a, b]).values
    Fp, R2p, pp = permanova(X[m], clean.country.values[m], perms=999)
    rows.append((a, b, int(m.sum()), Fp, R2p, pp))
pw = pd.DataFrame(rows, columns=["c1", "c2", "N", "F", "R2", "p_raw"])
order = np.argsort(pw.p_raw.values)
ranked = pw.p_raw.values[order]
adj = np.minimum.accumulate((ranked * len(pw) / np.arange(1, len(pw) + 1))[::-1])[::-1]
pw.loc[order, "p_FDR"] = np.minimum(adj, 1)
print(pw.round(3).to_string(index=False))

In [ ]:
# ---------- 2. exclude Bahrain ----------
print("\n=== 2. EXCLUDING BAHRAIN (repository-only countries) ===")
nb = clean[clean.country != "BH"].reset_index(drop=True)
Xn = zscore(nb[COLS].values)
F2, R22, p2 = permanova(Xn, nb.country.values, perms=9999)
Fd2, pd2 = permdisp(Xn, nb.country.values, perms=999, center="median")
print(f"n={len(nb)} {nb.country.value_counts().to_dict()}")
print(f"PERMANOVA pseudo-F={F2:.2f} R2={R22:.4f} p={p2:.4f} | PERMDISP F={Fd2:.3f} p={pd2:.3f}")
# also with the full-442 standardisation, to show robustness to scaling choice
Xn2 = X[(clean.country != "BH").values]
F2b, R22b, p2b = permanova(Xn2, nb.country.values, perms=9999)
print(f"(same, scaling from full 442) pseudo-F={F2b:.2f} R2={R22b:.4f} p={p2b:.4f}")

In [ ]:
# ---------- 3. balanced subsampling ----------
print("\n=== 3. BALANCED SUBSAMPLE (n=25 per country, 1000 repeats) ===")
n_min = clean.country.value_counts().min()
res = []
idx_by_c = {c: np.where(clean.country.values == c)[0] for c in clean.country.unique()}
for rep in range(1000):
    sel = np.concatenate([rng.choice(ix, n_min, replace=False) for ix in idx_by_c.values()])
    Xs = zscore(clean[COLS].values[sel])
    Fb, R2b, pb = permanova(Xs, clean.country.values[sel], perms=199, seed=rep)
    res.append((Fb, R2b, pb))
res = np.array(res)
print(f"n per country={n_min}; R2 median={np.median(res[:,1]):.4f} "
      f"(2.5-97.5%: {np.percentile(res[:,1],2.5):.4f}-{np.percentile(res[:,1],97.5):.4f}); "
      f"median pseudo-F={np.median(res[:,0]):.2f}; share p<0.05={np.mean(res[:,2]<0.05):.1%}; "
      f"median p={np.median(res[:,2]):.3f}")
 
# same, but balanced within repository-only countries (no Bahrain)
res_nb = []
idx_nb = {c: ix for c, ix in idx_by_c.items() if c != "BH"}
for rep in range(1000):
    sel = np.concatenate([rng.choice(ix, n_min, replace=False) for ix in idx_nb.values()])
    Xs = zscore(clean[COLS].values[sel])
    Fb, R2b, pb = permanova(Xs, clean.country.values[sel], perms=199, seed=rep)
    res_nb.append((Fb, R2b, pb))
res_nb = np.array(res_nb)
print(f"repository-only, n={n_min} each: R2 median={np.median(res_nb[:,1]):.4f}; "
      f"share p<0.05={np.mean(res_nb[:,2]<0.05):.1%}; median p={np.median(res_nb[:,2]):.3f}")

In [ ]:
# ---------- 4. one leaf per image ----------
print("\n=== 4. ONE LEAF PER IMAGE (1000 random draws) ===")
res1 = []
groups_by_key = clean.groupby("image_key").indices
keys = list(groups_by_key.keys())
for rep in range(1000):
    sel = np.array([rng.choice(groups_by_key[k]) for k in keys])
    Xs = zscore(clean[COLS].values[sel])
    F1, R21, p1 = permanova(Xs, clean.country.values[sel], perms=199, seed=rep)
    res1.append((F1, R21, p1, len(sel)))
res1 = np.array(res1)
print(f"n leaves={int(res1[0,3])}; R2 median={np.median(res1[:,1]):.4f} "
      f"(2.5-97.5%: {np.percentile(res1[:,1],2.5):.4f}-{np.percentile(res1[:,1],97.5):.4f}); "
      f"share p<0.05={np.mean(res1[:,2]<0.05):.1%}; median p={np.median(res1[:,2]):.3f}")